In [17]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'C:\work\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量 - 使用新的环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['HF_HUB_CACHE'] = cache_dir  
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline

# 新版本不再有 TRANSFORMERS_CACHE，可以从环境变量获取
actual_cache_dir = os.environ.get('HF_HUB_CACHE', '')
print(f"实际缓存路径: {actual_cache_dir}")

# 验证路径是否正确
if cache_dir == actual_cache_dir:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {actual_cache_dir}")

# 运行情感分析
try:
    classifier = pipeline("sentiment-analysis")
    result = classifier(
        [
            "I've been waiting for a HuggingFace course my whole life.",
            "I hate this so much!",
        ]
    )
    print(f"\n分析结果: {result}")
except Exception as e:
    print(f"运行pipeline时出错: {e}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

# 检查并修复缓存目录权限
import subprocess
try:
    subprocess.run(['icacls', cache_dir, '/grant', 'Everyone:F', '/T'], shell=True, capture_output=True)
    print("\n✅ 缓存目录权限设置完成")
except Exception as e:
    print(f"\n 权限设置失败: {e}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


设置的缓存目录: C:\work\Anaconda\envs\pytorch\cache
实际缓存路径: C:\work\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


运行pipeline时出错: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--bigcode--the-stack
  📁 datasets--code_search_net
  📁 datasets--conll2003
  📁 datasets--glue
  📁 datasets--imdb
  📁 datasets--lewtun--github-issues
  📁 datasets--liwu--MNBVC
  📁 datasets--pleisto--wikipedia-cn-20230720-filtered
  📁 datasets--wikitext
  📁 evaluate
  📁 metrics
  📁 models--bert-base-cased
  📁 models--bert-base-uncased
  📁 models--camembert-base
  📁 models--dbmdz--bert-large-cased-finetuned-conll03-english
  📁 models--distilbert--distilbert-base-cased-distilled-squad
  📁 models--distilbert--distilbert-base-unca

In [18]:
def load_with_encoding(filepath):
    encodings = ['gb18030', 'gbk', 'gb2312', 'utf-8', 'big5', 'ansi']
    
    for enc in encodings:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                content = f.read()
                print(f"成功使用编码: {enc}")
                return content
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise Exception("无法识别文件编码")

# 使用
content = load_with_encoding('2b-small_corpus.txt')
all_texts = [line.strip() for line in content.split('\n') if line.strip()]

成功使用编码: gb18030


In [4]:
# prepare_real_corpus.py
from datasets import load_dataset
import os
import random

def prepare_small_corpus():
    
    all_texts = []
    with open('2b-small_corpus.txt', 'r', encoding='gb18030') as f:
        content = f.read()
        # 按两个换行符分割成段落
        paragraphs = content.split('\n\n')
        for para in paragraphs:
            para = para.strip()
            if para:  # 跳过空段落
                all_texts.append(para)
    # 随机打乱
    random.shuffle(all_texts)
    
    # 划分数据集
    n = len(all_texts)
    train_texts = all_texts[:int(0.8*n)]
    val_texts = all_texts[int(0.8*n):int(0.9*n)]
    test_texts = all_texts[int(0.9*n):]
    
    # 保存
    os.makedirs('data', exist_ok=True)
    
    with open('data/train.txt', 'w', encoding='utf-8') as f:
        for text in train_texts:
            f.write(text.strip() + '\n\n')
    
    with open('data/val.txt', 'w', encoding='utf-8') as f:
        for text in val_texts:
            f.write(text.strip() + '\n\n')
    
    with open('data/test.txt', 'w', encoding='utf-8') as f:
        for text in test_texts:
            f.write(text.strip() + '\n\n')
    
    # 统计信息
    total_chars = sum(len(t) for t in all_texts)
    print(f"\n✅ 数据集准备完成!")
    print(f"   总样本数: {len(all_texts)}")
    print(f"   训练集: {len(train_texts)}")
    print(f"   验证集: {len(val_texts)}")
    print(f"   测试集: {len(test_texts)}")
    print(f"   总字符数: {total_chars:,}")
    
    return 'data/train.txt', 'data/val.txt', 'data/test.txt'

if __name__ == "__main__":
    prepare_small_corpus()


✅ 数据集准备完成!
   总样本数: 10117
   训练集: 8093
   验证集: 1012
   测试集: 1012
   总字符数: 1,129,932


In [ ]:
# transformer_working.py
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
import os
from tqdm import tqdm
import re

# ========== 1. Transformer模型 ==========
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        
        # 嵌入层
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        
        # 使用GPT风格的Transformer Decoder
        from transformers import GPT2Config, GPT2Model
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_layer=n_layer,
            n_head=n_head,
            n_positions=block_size,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout
        )
        self.transformer = GPT2Model(config)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # 权重初始化
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        # 位置编码由GPT2Model自动处理
        outputs = self.transformer(input_ids=idx)
        logits = self.lm_head(outputs.last_hidden_state)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=0)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, repetition_penalty=1.5):
        self.eval()
        
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            # 重复惩罚
            seq = idx[0].tolist()
            for token_id in set(seq[-20:]):
                if token_id != 0:
                    logits[0, token_id] /= repetition_penalty
            
            # 连续重复惩罚
            if len(seq) >= 2 and seq[-1] == seq[-2]:
                logits[0, seq[-1]] /= (repetition_penalty * 1.5)
            
            probs = F.softmax(logits, dim=-1)
            probs_top, indices = torch.topk(probs, top_k, dim=-1)
            probs_top = probs_top / probs_top.sum(dim=-1, keepdim=True)
            
            next_token = torch.multinomial(probs_top, 1)
            next_token = torch.gather(indices, -1, next_token)
            
            idx = torch.cat((idx, next_token), dim=1)
        
        return idx

# ========== 2. 数据集 ==========
class NovelDataset(Dataset):
    def __init__(self, data_path, tokenizer, block_size=256):
        self.tokenizer = tokenizer
        self.block_size = block_size
        
        with open(data_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        # 编码整个文本
        self.ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
        print(f"总token数: {len(self.ids)}")
        
        # 创建训练样本
        self.examples = []
        for i in range(0, len(self.ids) - block_size, block_size // 2):
            self.examples.append(self.ids[i:i+block_size])
        
        print(f"创建 {len(self.examples)} 个训练样本")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return torch.tensor(self.examples[idx], dtype=torch.long)

# ========== 3. 训练 ==========
def train():
    print("="*60)
    print("Transformer语言模型训练")
    print("="*60)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n使用设备: {device}")
    
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    print(f"词汇表大小: {tokenizer.vocab_size}")
    
    # 数据
    train_dataset = NovelDataset('data/train.txt', tokenizer, block_size=128)
    val_dataset = NovelDataset('data/val.txt', tokenizer, block_size=128)
    
    def collate_fn(batch):
        # 不需要额外padding，因为所有样本长度相同
        return torch.stack(batch)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
    
    # 模型
    model = TransformerLM(
        vocab_size=tokenizer.vocab_size,
        n_embd=384,
        n_head=6,
        n_layer=4,  # 4层更容易训练
        block_size=128,
        dropout=0.1
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=10000)
    
    print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
    print(f"训练样本: {len(train_dataset)}, 批次: {len(train_loader)}")
    
    best_loss = float('inf')
    
    for epoch in range(30):
        model.train()
        total_loss = 0
        progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/30')
        
        for batch in progress:
            batch = batch.to(device)
            targets = batch.clone()
            
            logits, loss = model(batch, targets)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            progress.set_postfix({'loss': f'{loss.item():.3f}'})
        
        avg_train_loss = total_loss / len(train_loader)
        
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                targets = batch.clone()
                logits, loss = model(batch, targets)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}: train={avg_train_loss:.3f}, val={avg_val_loss:.3f}, lr={scheduler.get_last_lr()[0]:.6f}")
        
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_transformer.pt')
            print(f"  ✅ 保存模型")
        
        # 测试生成
        if (epoch + 1) % 5 == 0:
            test_generation(model, tokenizer, device)

def test_generation(model, tokenizer, device):
    model.eval()
    
    prompts = ["今天天气", "我代表三体", "你好"]
    
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors='pt', add_special_tokens=False).to(device)
        
        with torch.no_grad():
            output = model.generate(
                input_ids, 
                max_new_tokens=50, 
                temperature=0.7, 
                top_k=40,
                repetition_penalty=1.5
            )
        
        generated = tokenizer.decode(output[0], skip_special_tokens=True)
        print(f"'{prompt}' -> {generated[:80]}...")

if __name__ == "__main__":
    train()


Transformer语言模型训练

使用设备: cuda
词汇表大小: 21128


Token indices sequence length is longer than the specified maximum sequence length for this model (886332 > 512). Running this sequence through the model will result in indexing errors


总token数: 886332
创建 13847 个训练样本
总token数: 105310
创建 1644 个训练样本
模型参数量: 31,536,384
训练样本: 13847, 批次: 866


Epoch 1/30: 100%|██████████| 866/866 [00:31<00:00, 27.71it/s, loss=0.016]


Epoch 1: train=0.583, val=0.015, lr=0.000461
  ✅ 保存模型


...

Epoch 29/30: 100%|██████████| 866/866 [00:31<00:00, 27.42it/s, loss=0.000]


Epoch 29: train=0.000, val=0.010, lr=0.000000


Epoch 30/30: 100%|██████████| 866/866 [00:31<00:00, 27.70it/s, loss=0.000]


Epoch 30: train=0.000, val=0.010, lr=0.000000
'今天天气' -> 今 天 天 气 气 气 气 气 记 记 记 导 导 导 存 存 票 票 负 负 负 车 车 车 他 他 岛 岛 虑 虑 虑 虑 形 形 床 床 床 床 场 场 ...
'我代表三体' -> 我 代 表 三 体 体 体 体 习 习 体 体 体 飞 飞 飞 飞 飞 飞 ， ， 走 走 走 走 走 画 画 远 远 远 远 争 争 争 从 从 从 从 从 ...
'你好' -> 你 好 好 渴 渴 失 失 失 失 失 滑 滑 滑 特 特 剧 剧 剧 剧 剧 一 一 一 一 淼 淼 海 海 丰 丰 攻 攻 攻 经 经 还 还 还 怕 怕 ...


RuntimeError: Error(s) in loading state_dict for TransformerLM:
	Missing key(s) in state_dict: "transformer.h.4.ln_1.weight", "transformer.h.4.ln_1.bias", "transformer.h.4.attn.c_attn.weight", "transformer.h.4.attn.c_attn.bias", "transformer.h.4.attn.c_proj.weight", "transformer.h.4.attn.c_proj.bias", "transformer.h.4.ln_2.weight", "transformer.h.4.ln_2.bias", "transformer.h.4.mlp.c_fc.weight", "transformer.h.4.mlp.c_fc.bias", "transformer.h.4.mlp.c_proj.weight", "transformer.h.4.mlp.c_proj.bias", "transformer.h.5.ln_1.weight", "transformer.h.5.ln_1.bias", "transformer.h.5.attn.c_attn.weight", "transformer.h.5.attn.c_attn.bias", "transformer.h.5.attn.c_proj.weight", "transformer.h.5.attn.c_proj.bias", "transformer.h.5.ln_2.weight", "transformer.h.5.ln_2.bias", "transformer.h.5.mlp.c_fc.weight", "transformer.h.5.mlp.c_fc.bias", "transformer.h.5.mlp.c_proj.weight", "transformer.h.5.mlp.c_proj.bias". 
	size mismatch for position_embedding.weight: copying a param with shape torch.Size([128

In [34]:
def generate_demo():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    
    # ✅ 必须使用与训练时相同的参数
    model = TransformerLM(
        vocab_size=tokenizer.vocab_size,
        n_embd=384,
        n_head=6,
        n_layer=4,
        block_size=128,  # 必须是128，不是256
        dropout=0.1
    ).to(device)
    
    model.load_state_dict(torch.load('best_transformer.pt', map_location=device))
    model.eval()
    
    print("\n" + "="*50)
    print("Transformer小说生成器")
    print("="*50)
    
    while True:
        prompt = input("\n请输入开头: ").strip()
        if prompt.lower() == 'quit':
            break
        
        input_ids = tokenizer.encode(prompt, return_tensors='pt', add_special_tokens=False).to(device)
        
        with torch.no_grad():
            output = model.generate(
                input_ids, 
                max_new_tokens=200, 
                temperature=0.7, 
                top_k=40,
                repetition_penalty=1.5
            )
        
        print(tokenizer.decode(output[0], skip_special_tokens=True))
        print("-"*50)

In [36]:
generate_demo()


Transformer小说生成器
你 好 我 是 地 球 指 挥 官 官 谋 谋 城 城 动 动 动 奏 奏 种 种 种 挂 挂 到 到 到 到 到 炬 炬 纷 纷 纷 活 活 银 银 固 固 固 固 准 准 网 网 助 助 其 其 主 主 启 启 启 状 状 个 个 个 过 过 过 过 过 过 写 写 写 于 于 迪 迪 讯 讯 呵 呵 门 门 门 门 门 门 门 执 执 执 选 选 狱 狱 狱 因 因 因 组 组 组 然 然 舰 舰 克 克 天 天 天 语 语 载 载 航 航 航 随 随 所 所 所 所 疗 疗 银 银 仰 仰 也 也 世 世 世 世 联 联 对 对 对 顿 顿 又 又 很 很 神 神 ， ， ， ， ， ， ， 头 头 头 头 头 米 米 登 登 都 都 澈 澈 季 季 季 悲 悲 积 积 积 岁 岁 散 散 余 余 余 复 复 复 一 一 一 居 居 平 平 山 山 山 原 原 原 兵 兵 他 他 杯 杯 航 航 握 握 握 握.
--------------------------------------------------
我 们 之 前 认 识 吗 吗 达 达 首 首 式 式 也 也 貌 貌 拿 拿 侧 侧 侧 侧 场 场 或 或 话 话 话 劝 劝 万 万 算 算 楚 楚 时 时 变 变 变 难 难 到 到 到 到 到 到 屏 屏 部 部 部 部 部 部 部 部 整 整 整 整 万 万 症 症 华 华 主 主 其 其 改 改 式 式 京 京 京 弃 弃 · · · 老 老 老 老 老 消 消 消 消 止 止 园 园 到 到 到 到 到 使 使 使 久 久 力 力 世 世 世 明 明 让 让 以 以 竟 竟 护 护 护 立 立 光 光 今 今 今 今 们 们 灭 灭 需 需 需 需 需 需 所 所 所 电 电 电 电 单 单 单 单 场 场 场 陶 陶 社 社 社 眠 眠 班 班 艾 艾 程 程 根 根 根 根 根 系 系 系 黑 黑 武 武 威 威 照 照 部 部 部 部 部 部 一 一 属 属 爆 爆 狄 狄 要 要 要 事 事
--------------------------------------------------
有 一 个 星 系 叫 三 体 体 仿 仿 显 显 显 显 显 壁 壁 壁 了 了 了 了 了 你